# Linear Algebra for Machine Learning
Companion to the linear algebra notes. Every result in the notes, checked numerically. Run the cells in order.

## 1. Data as a matrix, predictions as a matrix-vector product

In [ ]:
import numpy as np

X = np.array([[25, 70, 175], [30, 65, 168], [35, 80, 180], [40, 75, 170],
              [28, 58, 160], [45, 85, 178], [22, 55, 162]], dtype=float)
y = np.array([11.2, 12.1, 13.5, 14.3, 12.7, 15.0, 11.8])
w = np.array([0.20, 0.05, -0.01])

print("shape of X:", X.shape, " shape of X.T:", X.T.shape)
print("Priya, one dot product:", X[4] @ w + 5)
y_hat = X @ w + 5
print("all predictions:", np.round(y_hat, 2))
e = y - y_hat
print("errors:", np.round(e, 2), " sum of squared errors:", round(e @ e, 4))

### The column picture
$X w$ is the same as $w_1 c_1 + w_2 c_2 + w_3 c_3$ where $c_j$ are the columns of $X$.

In [ ]:
col_combo = w[0] * X[:, 0] + w[1] * X[:, 1] + w[2] * X[:, 2]
print(np.allclose(col_combo, X @ w))

## 2. Dot product geometry and norms

In [ ]:
a = np.array([3.5, 2.5]); b = np.array([5.0, 0.0])
cos_theta = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print("angle between a and b (degrees):", np.degrees(np.arccos(cos_theta)))

priya = X[4]
print("L1 norm:", np.abs(priya).sum(), "  L2 norm:", np.linalg.norm(priya))
# distances are dominated by the biggest-numbered feature (height) unless features are scaled
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print("distance John-Priya raw:", np.linalg.norm(X[0] - X[4]), " standardised:", np.linalg.norm(Z[0] - Z[4]))

## 3. Matrix multiplication: shapes and non-commutativity

In [ ]:
A = np.array([[1, 2], [3, 4], [5, 6]])
print((A @ A.T).shape, (A.T @ A).shape)
print("A @ A.T symmetric?", np.allclose(A @ A.T, (A @ A.T).T))
P = np.array([[1, 2], [3, 4]]); Q = np.array([[0, 1], [1, 0]])
print("PQ == QP ?", np.array_equal(P @ Q, Q @ P))

## 4. Determinants, inverses and rank

In [ ]:
A3 = np.array([[2, 1, 3], [0, 4, 1], [1, 2, 5]], dtype=float)
print("det (by hand we got 25):", np.linalg.det(A3))
A2 = np.array([[3, 6], [6, 14]], dtype=float)
print("inverse:\n", np.linalg.inv(A2))
B = np.array([[1, 2, 3], [2, 4, 6], [1, 0, 1]], dtype=float)
print("rank of B:", np.linalg.matrix_rank(B), " det of B:", np.linalg.det(B))
X4 = np.c_[X, X[:, 0] + X[:, 1]]          # a column that is age + weight
print("rank of sprint data with an Age+Weight column:", np.linalg.matrix_rank(X4), "of", X4.shape[1])

## 5. Coordinates in a non-orthogonal basis

In [ ]:
basis = np.array([[1, 2], [2, 3]], dtype=float).T   # columns are b1 = [1,2], b2 = [2,3]
c = np.linalg.solve(basis, np.array([5.0, 7.0]))
print("coordinates of [5, 7]:", c)    # expect [-1, 3]

## 6. Eigenvalues and eigenvectors

In [ ]:
A = np.array([[4, 1], [2, 3]], dtype=float)
vals, vecs = np.linalg.eig(A)
print("eigenvalues:", vals)
print("eigenvectors (columns, unit length):\n", vecs)
v1, v2 = np.array([1.0, 1.0]), np.array([1.0, -2.0])
print("A v1 =", A @ v1, " A v2 =", A @ v2)

# multiplying in the eigenbasis is just scaling
c = np.linalg.solve(np.c_[v1, v2], np.array([1.0, 2.0]))     # coordinates of [1, 2]
print("coordinates:", c, " -> A[1,2] =", 5 * c[0] * v1 + 2 * c[1] * v2, " direct:", A @ [1, 2])

## 7. Symmetric matrices: real eigenvalues, orthogonal eigenvectors

In [ ]:
S = np.array([[3, 1], [1, 3]], dtype=float)
vals, Q = np.linalg.eigh(S)
print("eigenvalues:", vals)
print("Q^T Q:\n", np.round(Q.T @ Q, 12))
print("Q diag(vals) Q^T == S ?", np.allclose(Q @ np.diag(vals) @ Q.T, S))

R = np.array([[0, -1], [1, 0]], dtype=float)   # a rotation: complex eigenvalues
print("rotation eigenvalues:", np.linalg.eigvals(R))

## 8. Projection: least squares and the hat matrix

In [ ]:
Xd = np.c_[np.ones(len(X)), X]
w_star, *_ = np.linalg.lstsq(Xd, y, rcond=None)
e = y - Xd @ w_star
print("best weights:", np.round(w_star, 4))
print("sum of squared errors:", round(e @ e, 4), "(our guessed weights gave 5.99)")
print("X^T e (should be ~0):", np.round(Xd.T @ e, 8))

H = Xd @ np.linalg.solve(Xd.T @ Xd, Xd.T)        # hat matrix
print("H symmetric:", np.allclose(H, H.T), " idempotent:", np.allclose(H @ H, H))

## 9. SVD and low-rank approximation

In [ ]:
M = np.array([[3, 2, 2], [2, 3, -2]], dtype=float)
U, s, Vt = np.linalg.svd(M)
print("singular values:", s)
rank1 = s[0] * np.outer(U[:, 0], Vt[0])
print("rank-1 approximation:\n", np.round(rank1, 4))
print("squared error of rank-1:", round(((M - rank1) ** 2).sum(), 4), " = sigma_2^2 =", s[1] ** 2)

# condition numbers: forming X^T X squares it
print("cond(X):", np.linalg.cond(Xd), " cond(X^T X):", np.linalg.cond(Xd.T @ Xd))

## Try it yourself
1. Predict Rajesh's sprint time by hand and compare with `y_hat[2]`.
2. Find the coordinates of `[4, 1]` in the basis `{[1,1], [1,-2]}` and compute `A @ [4,1]` using only eigenvalues.
3. Eigen-decompose `[[3,1],[1,3]]` by hand and compare with section 7.
4. Keep the first two singular values of the standardised sprint matrix `Z`. How much of the total squared size does that keep?